# Notebook 3 — Training and experiments

**Computer Vision CW1 — Diabetic Retinopathy Stage Detection**
Mohamed Shafran · NIBM (BSCCOMP24.2P)

This notebook trains the CNNs and runs the experiments. Each experiment answers one question, on the **validation** split only:

| ID | Question | What changes |
|---|---|---|
| E1 | How good is a plain transfer-learning baseline? | EfficientNetB0 at 224 px, variant A, no balancing |
| E2 | Does contrast/edge enhancement help? | variant B (CLAHE + unsharp) and C (Ben Graham) against A |
| E3 | Does class balancing help the rare stages? | class weights and oversampling against none |
| E4 | Is fine-tuning worth it, and at what learning rate? | the frozen-backbone result (phase 1) against the fine-tuned one (phase 2) of every run; fine-tuning learning rate 1e-5 against 1e-4 on the final configuration |
| E5 | Is a different backbone better? | EfficientNetB3 at 300 px and MobileNetV2 at 224 px |

Each step keeps the winner of the step before, so the notebook ends with one final model. **The test split is never touched here** — it is used once, in notebook 4.

**How to run:**
1. *Accelerator*: **GPU T4 x2** (one GPU is used) or GPU P100.
2. *Internet*: **On** (Settings → Internet), so Keras can download the ImageNet weights.
3. *Input*: only the output of notebook 2 (*Add Input → Your Work → Notebooks → dr-02-preprocess*).
4. **Save Version → Save & Run All**. It runs in the background for roughly 2–4 hours; you can close the tab.

A smoke test runs first (section 6). It sends a few images through every code path, so a mistake shows up in minutes, not hours.

In [ ]:
# ============================================================
# Setup: imports, folders, inputs, shared style
# ============================================================
import os, re, sys, json, time, shutil, zipfile, importlib, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from sklearn.metrics import confusion_matrix

# The GPU is for the training runs (separate processes, section 6). This notebook only draws
# figures and keeps score, so it runs its own TensorFlow work on the CPU.
gpus = tf.config.list_physical_devices("GPU")
print("GPUs available to the training runs:", [g.name for g in gpus] or "NONE - set the accelerator to a GPU")
tf.config.set_visible_devices([], "GPU")

OUT_DIR = Path("/kaggle/working")
FIG_DIR, RUN_DIR = OUT_DIR / "figures", OUT_DIR / "runs"
CACHE_DIR = Path("/tmp/dr_cache")                      # decoded images; not part of the output
for folder in (FIG_DIR, RUN_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
os.chdir(OUT_DIR)                                      # dr_training.py is written and run here

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
STAGE_COLORS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_SOFT, MUTED, GRID, FLAG = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#d03b3b"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK_SOFT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "font.size": 10, "figure.dpi": 110, "savefig.bbox": "tight",
})

def tidy(ax, grid_axis="y"):
    """Remove chart junk: drop the top/right frame, put a hairline grid behind the data."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
    return ax

def find_file(name, root="/kaggle/input"):
    """Search the attached inputs for a file, wherever Kaggle mounted it."""
    for path, dirs, files in os.walk(root):
        if name in files:
            return Path(path) / name
    raise FileNotFoundError(f"{name} not found under {root} - is notebook 2's output attached?")

# notebook 2's output: the split (with the camera columns), the three image archives, the module
SPLITS_CSV = find_file("splits.csv")
splits = pd.read_csv(SPLITS_CSV)
sys.path.insert(0, str(find_file("dr_preprocessing.py").parent))
import dr_preprocessing as prep                      # only its window mask and background level are used

y_all = splits["diagnosis"].to_numpy()
idx_train = np.where(splits["split"] == "train")[0]
idx_val = np.where(splits["split"] == "val")[0]
print("images:", len(splits), "| train", len(idx_train), "| val", len(idx_val),
      "| test", int((splits["split"] == "test").sum()), "(not used here)")

## 1. The training code, and the settings

All the training code lives in one file, **`dr_training.py`**, written by the next cell. The notebook runs that file **once per training run, each time as a fresh Python process**, and collects the results. The reason is memory. TensorFlow does not always hand back all the memory of a model that has been deleted: in a test, every training run left about 0.3 GB behind, which over 13 runs could exhaust the machine. A new process starts clean every time, and a crash in one run cannot damage the results of the others.

| Setting | Value | Reason |
|---|---|---|
| Backbone | EfficientNetB0, 224 px | Small (about 4 M parameters without its ImageNet classifier), fast, strong on ImageNet; B3 and MobileNetV2 are tested in E5 |
| Head | global average pooling → dropout 0.3 → 5 softmax outputs | Few new weights to learn from 2,441 images |
| Phase 1 | backbone frozen, Adam, learning rate 1e-3 (Adam's default), up to 8 epochs | The random new head must settle before it sends gradients into the backbone |
| Phase 2 | top 25 % of the backbone unfrozen, learning rate 1e-4 (1e-5 is tried in E4), up to 20 epochs | Adapts the high-level features to retinas; the lower layers (edges, textures) stay as ImageNet learned them |
| Batch size | 32 (16 for B3 at 300 px) | Fits a T4's memory |
| Model selection | best **validation macro-F1** of the system's answer | Accuracy rewards ignoring the rare stages; macro-F1 does not |
| Early stopping | patience 3 (phase 1), 5 (phase 2), best weights kept | Stops when validation macro-F1 stops improving |
| Learning-rate schedule | halve after 2 epochs without improvement | Finer steps once progress stalls |
| Seeds | 42 and 43 (B3: 42 only, for time) | Training twice shows how much of a difference is luck |

**Why two seeds.** A network starts from random weights and sees the images in a random order, so two runs with identical settings finish at slightly different scores. With only 523 validation images, a gap of 0.01–0.02 in macro-F1 can be pure chance. Running each setting twice, and showing the spread, stops the notebook from "discovering" differences that are noise.

**The system's answer (the two-step rule from the design).** The model outputs five probabilities. *DR present* if the probabilities of stages 1–4 add up to 0.5 or more; the stage is then the most likely of stages 1–4, otherwise stage 0. Every metric below is computed on that answer. The plain "highest probability" answer is also recorded, to check that the rule costs nothing.

In [ ]:
%%writefile dr_training.py
"""
dr_training.py - trains ONE configuration and scores it on the validation split.

Notebook 3 runs this file once per training run, each time in a fresh Python process:
    python dr_training.py '{"name": "E1_A_s42", "backbone": "EfficientNetB0", ...}'
Why a fresh process: TensorFlow does not always hand back all the memory of a model that has
been deleted, so thirteen runs inside one process would pile it up. A new process starts
clean, and a crash in one run cannot damage the results of the others.

The notebook also imports this file for its shared pieces (augmentation, model, metrics),
so the figures and tables are made with exactly the same code as the training.

Computer Vision CW1 - Diabetic Retinopathy Stage Detection - Mohamed Shafran
"""
import os
import sys
import json
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.ticker
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

tf.get_logger().setLevel("ERROR")                        # hide TensorFlow's retracing notices

# ---------------------------------------------------------------------------
# Settings (notebook 3, section 1 explains each one)
# ---------------------------------------------------------------------------
WEIGHTS = "imagenet"
SIZES = {"EfficientNetB0": 224, "EfficientNetB3": 300, "MobileNetV2": 224}
BATCH = {"EfficientNetB0": 32, "EfficientNetB3": 16, "MobileNetV2": 32}
HEAD_EPOCHS, HEAD_LR, HEAD_PATIENCE = 8, 1e-3, 3           # phase 1: backbone frozen
FT_EPOCHS, FT_LR, FT_PATIENCE = 20, 1e-4, 5               # phase 2: top of the backbone fine-tuned
UNFREEZE_FRACTION = 0.25
DROPOUT = 0.3

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
BLUE, ORANGE, INK, INK_SOFT, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"


# ---------------------------------------------------------------------------
# Augmentation: Keras layers that are active ONLY while training
# ---------------------------------------------------------------------------
def make_augmenter(fill):
    """Random flips, any rotation, zoom, shift, brightness and contrast.
    `fill` is the background value for areas uncovered by rotating or shifting."""
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.5, fill_mode="constant", fill_value=fill),        # 0.5 turn = +-180 deg
        layers.RandomZoom(0.1, fill_mode="constant", fill_value=fill),
        layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=fill),
        layers.RandomBrightness(0.15, value_range=(0, 255)),
        layers.RandomContrast(0.15, value_range=(0, 255)),
    ], name="augment")


# ---------------------------------------------------------------------------
# The model
# ---------------------------------------------------------------------------
def build_model(backbone, size, fill, weights="default"):
    """Input (0-255 RGB) -> augmentation -> pretrained backbone -> GAP -> dropout -> 5 probabilities.
    The backbone is built on the augmentation's output rather than nested as one layer, so its
    last convolution layer stays reachable for Grad-CAM. Returns the model, the backbone's own
    layers (for freezing) and the name of its last convolution layer."""
    weights = WEIGHTS if weights == "default" else weights     # None = random start (tests only)
    inputs = keras.Input((size, size, 3), name="image")
    x = make_augmenter(fill)(inputs)
    if backbone == "MobileNetV2":                        # MobileNetV2 expects -1..1, EfficientNet 0..255
        x = layers.Rescaling(1 / 127.5, offset=-1.0, name="to_minus1_plus1")(x)
        base = keras.applications.MobileNetV2(include_top=False, weights=weights, input_tensor=x)
        last_conv = "out_relu"
    else:
        application = getattr(keras.applications, backbone)          # EfficientNetB0 / EfficientNetB3
        base = application(include_top=False, weights=weights, input_tensor=x)
        last_conv = "top_activation"
    y = layers.GlobalAveragePooling2D(name="gap")(base.output)
    y = layers.Dropout(DROPOUT, name="dropout")(y)
    outputs = layers.Dense(5, activation="softmax", name="stage_probs")(y)
    model = keras.Model(inputs, outputs, name=f"dr_{backbone}")

    ours = {"image", "augment", "to_minus1_plus1"}                    # layers before the backbone
    backbone_layers = [layer for layer in base.layers if layer.name not in ours]
    return model, backbone_layers, last_conv


def set_trainable(backbone_layers, top_fraction):
    """Freeze the backbone, then unfreeze its top `top_fraction` of layers - except
    BatchNormalization, which stays frozen (inference mode) so its ImageNet statistics survive."""
    first_unfrozen = len(backbone_layers) - int(round(len(backbone_layers) * top_fraction))
    for i, layer in enumerate(backbone_layers):
        layer.trainable = i >= first_unfrozen and not isinstance(layer, layers.BatchNormalization)


# ---------------------------------------------------------------------------
# Data and class balancing (training split only)
# ---------------------------------------------------------------------------
def class_weights(y_train):
    """'balanced' weights from the TRAINING labels: n_images / (5 x n_images_of_that_stage)."""
    return dict(enumerate(compute_class_weight("balanced", classes=np.arange(5), y=y_train)))


def make_train_dataset(X, y, idx, batch, balance, seed):
    """tf.data pipeline for the training images: shuffle (or oversample), batch, cast, prefetch.
    Returns the dataset and the number of batches in one epoch."""
    def as_float(images, labels):
        return tf.cast(images, tf.float32), labels
    if balance == "oversample":                          # one endless stream per stage, drawn equally
        streams = []
        for stage in range(5):
            own = idx[y[idx] == stage]
            streams.append(tf.data.Dataset.from_tensor_slices((X[own], y[own]))
                           .shuffle(len(own), seed=seed).repeat())
        ds = tf.data.Dataset.sample_from_datasets(streams, weights=[0.2] * 5, seed=seed)
    else:
        ds = tf.data.Dataset.from_tensor_slices((X[idx], y[idx])).shuffle(len(idx), seed=seed).repeat()
    steps = int(np.ceil(len(idx) / batch))
    ds = ds.batch(batch).map(as_float, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    return ds, steps


# ---------------------------------------------------------------------------
# The system's answer and the metrics
# ---------------------------------------------------------------------------
def two_step(probs):
    """DR present if P(stages 1-4) >= 0.5; then the most likely of stages 1-4; otherwise stage 0."""
    has_dr = probs[:, 1:].sum(axis=1) >= 0.5
    return np.where(has_dr, 1 + probs[:, 1:].argmax(axis=1), 0)


def scores(true, pred):
    """Metrics for the 5 stages and for DR yes/no (the same definitions as notebook 2)."""
    true, pred = np.asarray(true), np.asarray(pred)
    dr_true, dr_pred = true > 0, pred > 0
    return {
        "accuracy": float(accuracy_score(true, pred)),
        "macro_f1": float(f1_score(true, pred, labels=list(range(5)), average="macro", zero_division=0)),
        "qwk": float(cohen_kappa_score(true, pred, weights="quadratic")),
        "dr_accuracy": float(accuracy_score(dr_true, dr_pred)),
        "dr_sensitivity": float(recall_score(dr_true, dr_pred, zero_division=0)),
        "dr_specificity": float(recall_score(~dr_true, ~dr_pred, zero_division=0)),
    }


class ValMetrics(keras.callbacks.Callback):
    """After every epoch: validation macro-F1, QWK and DR accuracy of the two-step answer, added to
    the logs so that the callbacks listed after this one (checkpoint, early stopping, learning-rate
    schedule, CSV log) can use them."""
    def __init__(self, X_val, y_val, batch_size):
        super().__init__()
        self.X_val, self.y_val, self.batch_size = X_val, y_val, batch_size

    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.X_val, batch_size=self.batch_size, verbose=0)
        s = scores(self.y_val, two_step(probs))
        logs["val_macro_f1"], logs["val_qwk"], logs["val_dr_accuracy"] = s["macro_f1"], s["qwk"], s["dr_accuracy"]
        print(f"   epoch {epoch + 1:>2}: loss {logs['loss']:.3f} | val loss {logs['val_loss']:.3f} | "
              f"val macro-F1 {s['macro_f1']:.3f} | QWK {s['qwk']:.3f} | DR acc {s['dr_accuracy']:.3f}",
              flush=True)


def plot_curves(history, split_epoch, title, path, show=False):
    """Training vs validation accuracy and loss over both phases; dashed line = fine-tuning starts."""
    epochs = np.arange(1, len(history) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="#fcfcfb")
    for ax, key, label in ((axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")):
        ax.set_facecolor("#fcfcfb")
        ax.plot(epochs, history[key], color=BLUE, linewidth=2, label="training")
        ax.plot(epochs, history[f"val_{key}"], color=ORANGE, linewidth=2, label="validation")
        ax.axvline(split_epoch + 0.5, color=MUTED, linestyle="--", linewidth=1)
        ax.text(split_epoch + 0.6, ax.get_ylim()[1], " fine-tuning", va="top", fontsize=8.5, color=INK_SOFT)
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))   # whole epochs only
        ax.ticklabel_format(axis="y", useOffset=False)
        ax.set_xlabel("Epoch", color=INK_SOFT)
        ax.set_ylabel(label, color=INK_SOFT)
        ax.set_title(label, fontsize=12, fontweight="bold")
        ax.legend(frameon=False, labelcolor=INK_SOFT)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.grid(axis="y", color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    fig.suptitle(title, fontsize=12, fontweight="bold", color=INK)
    plt.tight_layout(rect=(0, 0, 1, 0.94))
    plt.savefig(path, dpi=110, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)


# ---------------------------------------------------------------------------
# One training run
# ---------------------------------------------------------------------------
def train_run(cfg, X, splits, smoke=False):
    """Phase 1 (backbone frozen) then phase 2 (top of the backbone fine-tuned); reload the best
    checkpoint of both phases, score it on the validation split, save everything in run_dir.
    `smoke=True` shrinks the run to a few images and two batches per phase."""
    run_dir = cfg["run_dir"]
    os.makedirs(run_dir, exist_ok=True)
    keras.utils.set_random_seed(cfg["seed"])
    start = time.time()

    y = splits["diagnosis"].to_numpy()
    idx_train = np.where(splits["split"] == "train")[0]
    idx_val = np.where(splits["split"] == "val")[0]
    if smoke:                                            # 12 images per stage, 24 validation images
        idx_train = np.concatenate([idx_train[y[idx_train] == s][:12] for s in range(5)])
        idx_val = idx_val[:24]
    size, batch = SIZES[cfg["backbone"]], BATCH[cfg["backbone"]]
    X_val, y_val = X[idx_val].astype("float32"), y[idx_val]
    fill = 128.0 if cfg["variant"] == "C" else 0.0       # background of variant C is grey, else black

    ft_lr = float(cfg.get("ft_lr", FT_LR))               # experiment E4 also tries 1e-5
    model, backbone_layers, _ = build_model(cfg["backbone"], size, fill)
    train_ds, epoch_steps = make_train_dataset(X, y, idx_train, batch, cfg["balance"], cfg["seed"])
    weights = class_weights(y[idx_train]) if cfg["balance"] == "weights" else None
    checkpoint = keras.callbacks.ModelCheckpoint(os.path.join(run_dir, "best.keras"),
                                                 monitor="val_macro_f1", mode="max",
                                                 save_best_only=True)      # shared by both phases

    def fit(epochs, learning_rate, patience, phase):
        model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        callbacks = [ValMetrics(X_val, y_val, batch), checkpoint,
                     keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max",
                                                   patience=patience, restore_best_weights=True),
                     keras.callbacks.ReduceLROnPlateau(monitor="val_macro_f1", mode="max",
                                                       factor=0.5, patience=2, min_lr=1e-7),
                     keras.callbacks.CSVLogger(os.path.join(run_dir, f"history_phase{phase}.csv"))]
        return model.fit(train_ds, steps_per_epoch=2 if smoke else epoch_steps, epochs=epochs,
                         validation_data=(X_val, y_val), class_weight=weights,
                         callbacks=callbacks, verbose=0)

    print(f"--- {cfg['name']}: {cfg['backbone']} {size} px, variant {cfg['variant']}, "
          f"balance {cfg['balance']}, fine-tuning lr {ft_lr:g}, seed {cfg['seed']}", flush=True)
    print(" phase 1 - backbone frozen, head learning", flush=True)
    set_trainable(backbone_layers, 0.0)
    h1 = fit(1 if smoke else HEAD_EPOCHS, HEAD_LR, HEAD_PATIENCE, 1)
    print(" phase 2 - top 25% of the backbone fine-tuned", flush=True)
    set_trainable(backbone_layers, UNFREEZE_FRACTION)
    h2 = fit(1 if smoke else FT_EPOCHS, ft_lr, FT_PATIENCE, 2)

    history = pd.concat([pd.DataFrame(h1.history), pd.DataFrame(h2.history)], ignore_index=True)
    history.insert(0, "phase", [1] * len(h1.history["loss"]) + [2] * len(h2.history["loss"]))
    best = keras.models.load_model(os.path.join(run_dir, "best.keras"))
    probs = best.predict(X_val, batch_size=batch, verbose=0)
    answer = two_step(probs)

    result = {**cfg, "size": size, "ft_lr": ft_lr, **scores(y_val, answer),
              "macro_f1_argmax": float(f1_score(y_val, probs.argmax(axis=1), labels=list(range(5)),
                                                average="macro", zero_division=0)),
              "phase1_best_macro_f1": float(max(h1.history["val_macro_f1"])),
              "best_epoch": int(np.argmax(history["val_macro_f1"].to_numpy()) + 1),
              "epochs_phase1": len(h1.history["loss"]), "epochs_phase2": len(h2.history["loss"]),
              "minutes": round((time.time() - start) / 60, 1)}
    mixed = splits.iloc[idx_val]["camera_group"].to_numpy() == cfg.get("mixed_camera")
    if mixed.sum() >= 5:                                 # the camera that gives the least away
        mixed_scores = scores(y_val[mixed], answer[mixed])
        result["mixed_camera_macro_f1"] = mixed_scores["macro_f1"]
        result["mixed_camera_dr_accuracy"] = mixed_scores["dr_accuracy"]
    if not smoke:
        history.to_csv(os.path.join(run_dir, "history.csv"), index=False)
        pd.DataFrame({"id_code": splits.iloc[idx_val]["id_code"].to_numpy(), "true": y_val, "pred": answer,
                      **{f"p{k}": probs[:, k] for k in range(5)}}
                     ).to_csv(os.path.join(run_dir, "val_predictions.csv"), index=False)
        plot_curves(history, len(h1.history["loss"]), cfg["name"], os.path.join(run_dir, "curves.png"))
    with open(os.path.join(run_dir, "result.json"), "w") as f:
        json.dump(result, f, indent=2)
    print(f" => val macro-F1 {result['macro_f1']:.3f} | QWK {result['qwk']:.3f} | "
          f"DR accuracy {result['dr_accuracy']:.3f} | {result['minutes']} min", flush=True)
    return result


def main(cfg_json):
    """Entry point for `python dr_training.py '<json>'`: load the split and the cached images, train."""
    matplotlib.use("Agg")                                # a background process draws to files only
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
    cfg = json.loads(cfg_json)
    splits = pd.read_csv(cfg["splits_csv"])
    X = np.load(cfg["cache"])                            # uint8 array, rows in splits.csv order
    train_run(cfg, X, splits, smoke=cfg.get("smoke", False))


if __name__ == "__main__":
    main(sys.argv[1])


In [ ]:
import dr_training as T
importlib.reload(T)                    # picks up changes if the cell above is edited and re-run

SEEDS = {"EfficientNetB0": [42, 43], "EfficientNetB3": [42], "MobileNetV2": [42, 43]}
MIN_BASELINE_F1 = 0.30                 # E1 must beat this, or the run stops (section 7)
MIXED_CAMERA = "2588x1958"             # 44% of its images are healthy - the camera gives the least away

print("sizes:", T.SIZES, "| batch:", T.BATCH)
print(f"phase 1: {T.HEAD_EPOCHS} epochs max, lr {T.HEAD_LR} | phase 2: {T.FT_EPOCHS} epochs max, "
      f"lr {T.FT_LR}, top {T.UNFREEZE_FRACTION:.0%} unfrozen | dropout {T.DROPOUT}")

## 2. Decode the cached images once

Notebook 2 saved every image as a 512 × 512 PNG inside a zip. Here each variant is decoded **once**, resized to the model's input size with area interpolation, and saved as a single array file on the machine's local disk. Every training run then loads that file in a second or two, instead of decoding thousands of PNGs. A variant at 224 px is about 0.5 GB.

The same pass counts the **uneven cut-off flaw** from notebook 2 (figure 14, stage-4 row): photos where the window's corners reach past the retina, leaving near-black pixels along the window's edge.

In [ ]:
EDGE_RING = prep.MASK & ~cv2.erode(prep.MASK.astype(np.uint8), np.ones((25, 25), np.uint8)).astype(bool)
dark_edge_pixels = None                        # filled the first time variant A is decoded


def decode_images(variant, size, ids):
    """Decode the PNGs of `ids` from notebook 2's zip, resize to `size`, count near-black
    pixels on the window's edge. Returns (images, dark-edge counts)."""
    with zipfile.ZipFile(find_file(f"preprocessed_{variant}.zip")) as archive:
        blobs = [archive.read(f"{c}.png") for c in ids]

    def decode(blob):
        img = cv2.cvtColor(cv2.imdecode(np.frombuffer(blob, np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        dark = int(((cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) <= prep.BG_TOL) & EDGE_RING).sum())
        if size != img.shape[0]:
            img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
        return img, dark

    with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
        decoded = list(pool.map(decode, blobs))
    return np.stack([img for img, _ in decoded]), [dark for _, dark in decoded]


def cache_variant(variant, size):
    """Decode one variant for every image (splits.csv order) and save it as one .npy file."""
    global dark_edge_pixels
    path = CACHE_DIR / f"{variant}_{size}.npy"
    if not path.exists():
        start = time.time()
        X, dark = decode_images(variant, size, splits["id_code"])
        np.save(path, X)
        if variant == "A" and dark_edge_pixels is None:
            dark_edge_pixels = pd.Series(dark, index=splits["id_code"])
        print(f"variant {variant} at {size} px: {len(X)} images, {X.nbytes / 1e9:.2f} GB, "
              f"{time.time() - start:.0f} s")
        del X
    return path


for variant in "ABC":                          # E1 and E2 need all three at 224 px
    cache_variant(variant, T.SIZES["EfficientNetB0"])
if dark_edge_pixels is None:                   # cache already existed (cell re-run): count again
    dark_edge_pixels = pd.Series(decode_images("A", 32, splits["id_code"])[1], index=splits["id_code"])

flawed = dark_edge_pixels[dark_edge_pixels >= 100]
print(f"\nphotos with the uneven cut-off flaw (>= 100 near-black pixels on the window edge): "
      f"{len(flawed)} of {len(splits)} ({100 * len(flawed) / len(splits):.1f}%)")
print("by camera:", splits.set_index("id_code").loc[flawed.index, "camera_group"].value_counts().to_dict())

In [ ]:
# Figure 19 - the flagged photos, so the count above can be checked by eye
strongest = flawed.sort_values(ascending=False).index[:6]
if len(strongest):
    shown, _ = decode_images("A", prep.SIZE, strongest)
    fig, axes = plt.subplots(1, len(shown), figsize=(3 * len(shown), 3.4), squeeze=False)
    for ax, img, code_id in zip(axes[0], shown, strongest):
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"{code_id}\n{int(flawed[code_id])} dark edge px", fontsize=9, color=INK_SOFT)
    fig.suptitle(f"Uneven cut-off: {len(flawed)} photos flagged (the {len(shown)} strongest shown)",
                 fontsize=12, fontweight="bold", color=INK)
    plt.tight_layout(rect=(0, 0, 1, 0.9))
    plt.savefig(FIG_DIR / "fig19_uneven_cutoff.png")
    plt.show()
else:
    print("no photo flagged - nothing to show")

## 3. Augmentation — only while training

**Additional knowledge: Keras preprocessing layers.** In your practicals, augmentation came from `ImageDataGenerator`, which current Keras marks as deprecated. Here the augmentation is a small block of Keras layers at the start of the model (`make_augmenter` in `dr_training.py`). These layers are **active only during training**. When the model predicts on validation or test images, or in the app, they pass the image through untouched, so evaluation images are never augmented. Running them inside the model also lets the GPU do the work.

| Transform | Range | Why it is realistic for fundus photos |
|---|---|---|
| Flip horizontal and vertical | — | Left and right eyes are mirror images, and a fundus photo has no fixed "up" |
| Rotation | any angle | The camera can be turned in any direction relative to the eye |
| Zoom | ±10 % | Retinas appear at slightly different sizes |
| Shift | ±5 % | The eye is never perfectly centred |
| Brightness, contrast | ±15 % | Imitates exposure differences between cameras and clinics |

**Left out on purpose:**
- Colour (hue) shifts, because the colour of lesions carries meaning — yellow exudates, red haemorrhages — and Ben Graham found colour augmentation did not help.
- *Cutout* (blanking out a patch), because it could erase the only lesion in a mild case.

Areas uncovered by rotating or shifting are filled with the variant's background: black for A and B, grey 128 for C. The brightness change also lifts the black background a little in some training images; that is random and unrelated to the stage, so it teaches the model to ignore the background.

In [ ]:
# Figure 20 - one training image and eight augmented versions of it
tf.keras.utils.set_random_seed(42)
augmenter = T.make_augmenter(0.0)
sample = np.load(CACHE_DIR / f"A_{T.SIZES['EfficientNetB0']}.npy", mmap_mode="r")[idx_train[0]].astype("float32")
augmented = augmenter(np.repeat(sample[None], 8, axis=0), training=True).numpy()

fig, axes = plt.subplots(3, 3, figsize=(8.4, 8.8))
for k, (ax, img) in enumerate(zip(axes.ravel(), [sample] + list(augmented))):
    ax.imshow(np.clip(img, 0, 255).astype("uint8"))
    ax.axis("off")
    ax.set_title("original (variant A)" if k == 0 else f"augmented {k}", fontsize=9, color=INK_SOFT)
fig.suptitle("Training-time augmentation: flips, any rotation, zoom, shift, brightness, contrast",
             fontsize=12, fontweight="bold", color=INK)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.savefig(FIG_DIR / "fig20_augmentation.png")
plt.show()

## 4. Class balancing — only on the training split

Severe (stage 3) has about 1 image for every 9 of No DR. A model can score well by rarely predicting the rare stages, so experiment E3 compares three options:

- **None** — the images in their natural proportions.
- **Class weights** *(in your practicals)* — an error on a rare stage counts more in the loss. The weights come from the **training** split only: `weight = n_images / (5 × n_images_of_that_stage)`.
- **Oversampling** *(additional knowledge)* — rare-stage images are drawn more often, so each stage makes up a fifth of every epoch. Augmentation changes every copy, so a repeated image never looks exactly the same twice.

The two methods are never combined, which would count the imbalance twice. The validation split always keeps its real proportions, so its scores stay honest.

In [ ]:
# Figure 21 - what each option does to one epoch
weights = T.class_weights(y_all[idx_train])
print("class weights from the training split:", {CLASS_NAMES[k]: round(float(v), 2) for k, v in weights.items()})
counts = np.bincount(y_all[idx_train], minlength=5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.5, 1]})
x, width = np.arange(5), 0.36
for k, (label, values, colour) in enumerate([("natural (none, class weights)", counts, MUTED),
                                             ("oversampling", np.full(5, counts.sum() / 5), BLUE)]):
    bars = axes[0].bar(x + (k - 0.5) * width, values, width=width - 0.04, color=colour, label=label)
    for bar, v in zip(bars, values):
        axes[0].text(bar.get_x() + bar.get_width() / 2, v + counts.max() * 0.015, f"{v:.0f}",
                     ha="center", fontsize=8.5, color=INK_SOFT)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{i}\n{CLASS_NAMES[i]}" for i in range(5)])
axes[0].set_ylabel("Images per stage in one epoch")
axes[0].set_title("Oversampling shows every stage equally often")
axes[0].legend(frameon=False, labelcolor=INK_SOFT, loc="upper right")
tidy(axes[0])
w = [weights[k] for k in range(5)]
bars = axes[1].bar(x, w, width=0.6, color=STAGE_COLORS)
for bar, v in zip(bars, w):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + max(w) * 0.02, f"{v:.2f}",
                 ha="center", fontsize=9, color=INK_SOFT)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{i}\n{CLASS_NAMES[i]}" for i in range(5)])
axes[1].set_ylabel("Weight of one image in the loss")
axes[1].set_title(f"Class weights: a Severe error counts {w[3] / w[0]:.1f}x a No DR error")
tidy(axes[1])
plt.tight_layout()
plt.savefig(FIG_DIR / "fig21_balancing.png")
plt.show()

## 5. The model and transfer learning

**EfficientNet** *(additional knowledge: compound scaling)*. The EfficientNet family grows one small network in depth, width and input resolution together, in fixed proportions. B0 is the smallest: about 4 M parameters once its ImageNet classifier is removed (5.3 M with it), at 224 px. B3 is larger (about 11 M without the classifier) and sees 300 px images, which can matter for tiny lesions. The cell below prints the exact counts for this model. Keras's EfficientNet contains its own rescaling, so it takes pixels in the **0–255** range as they are. Feeding it 0–1 images was one of the bugs that broke the lecturer's notebook. MobileNetV2, from your transfer-learning practical, expects −1 to 1, so a rescaling layer is added in front of it.

**Built "flat", for Grad-CAM later.** The backbone is built on top of the augmentation block's output, not nested inside the model as a single layer. Its last convolution layer can then be reached directly from the full model. The lecturer's notebook failed at Grad-CAM with *"No convolutional layer found"* because its backbone was nested.

**Two phases.** Phase 1 freezes the whole backbone and trains only the new head. Phase 2 unfreezes the top 25 % of the backbone's layers and continues with a 10× smaller learning rate.

**BatchNormalization stays frozen in phase 2** *(additional knowledge)*. BatchNorm layers keep running averages of their inputs, learned on millions of ImageNet photos. If they updated on small batches of retinas, those averages would jump around and damage what the network already knows. Keras runs a BatchNorm layer whose `trainable` is False in inference mode, which is the official advice for fine-tuning.

In [ ]:
# the three backbones, once, to report their sizes (this also downloads the ImageNet weights)
for name, size in T.SIZES.items():
    model, backbone_layers, last_conv = T.build_model(name, size, 0.0)
    T.set_trainable(backbone_layers, 0.0)
    phase1 = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    T.set_trainable(backbone_layers, T.UNFREEZE_FRACTION)
    phase2 = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    total = sum(int(np.prod(w.shape)) for w in model.weights)
    print(f"{name:<15} {size} px | {total / 1e6:4.1f} M parameters | trainable: phase 1 {phase1:>9,} "
          f"-> phase 2 {phase2:>10,} | Grad-CAM layer: {model.get_layer(last_conv).name}")
    del model
tf.keras.backend.clear_session()

## 6. Running one configuration

`train_run` in `dr_training.py` does everything for one setting:

1. Builds a fresh model, with the seed fixed.
2. **Phase 1**: trains the head with the backbone frozen.
3. **Phase 2**: unfreezes the top of the backbone and fine-tunes it.
4. Reloads the best checkpoint across both phases, scores it on the validation split and saves everything in `runs/<name>/`.

**Additional knowledge — a custom callback.** Keras callbacks run code at the end of every epoch. The callbacks you know — EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger — can only watch numbers Keras already computes, such as accuracy. `ValMetrics` computes the validation **macro-F1**, **QWK** and **DR yes/no accuracy** of the two-step answer after every epoch and hands them to the other callbacks. Early stopping and checkpointing then follow macro-F1 instead of accuracy.

`run_child` starts `dr_training.py` as a separate process and shows its log line by line.

The **smoke test** trains every combination the notebook will use, for two batches on a handful of images, so a mistake stops the run within minutes.

In [ ]:
NOISE = re.compile(r"^(E0000|W0000|I0000|WARNING: All log messages)|Unable to register|oneDNN custom|"
                   r"cpu_feature_guard|To enable the following|rebuild TensorFlow|TF-TRT|computation_placer")

def run_child(cfg):
    """Run dr_training.py for one configuration in a fresh Python process, printing its log."""
    env = {**os.environ, "TF_CPP_MIN_LOG_LEVEL": "2", "PYTHONUNBUFFERED": "1"}
    process = subprocess.Popen([sys.executable, "dr_training.py", json.dumps(cfg)], cwd=OUT_DIR, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        if not NOISE.search(line):
            print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"training run {cfg['name']} failed - the error is in the log above")


# Smoke test: every combination the experiments will use, on 12 images per stage + 24 validation images
start = time.time()
tiny = np.concatenate([np.concatenate([idx_train[y_all[idx_train] == s][:12] for s in range(5)]), idx_val[:24]])
tiny_splits = splits.iloc[tiny].reset_index(drop=True)
tiny_csv = CACHE_DIR / "smoke_splits.csv"
tiny_splits.to_csv(tiny_csv, index=False)
checks = [("EfficientNetB0", "A", "none"), ("EfficientNetB0", "B", "weights"),
          ("EfficientNetB0", "C", "oversample"), ("MobileNetV2", "A", "none"),
          ("EfficientNetB3", "C", "none")]
for backbone, variant, balance in checks:
    cache = CACHE_DIR / f"smoke_{variant}_{T.SIZES[backbone]}.npy"
    np.save(cache, decode_images(variant, T.SIZES[backbone], tiny_splits["id_code"])[0])
    name = f"smoke_{backbone}_{variant}_{balance}"
    run_child({"name": name, "backbone": backbone, "variant": variant, "balance": balance, "seed": 42,
               "smoke": True, "run_dir": str(CACHE_DIR / name), "cache": str(cache),
               "splits_csv": str(tiny_csv), "mixed_camera": MIXED_CAMERA})
print(f"\nSMOKE TEST PASSED - every path works ({time.time() - start:.0f} s)")

## 7. Run the experiments

Each block below trains its configurations, prints a comparison and passes the winner on to the next step. The rule is always the same: **highest mean validation macro-F1 across seeds**. When the leader beats the simpler (or current) option by less than the spread between seeds, the difference is treated as noise and that option is kept: A over B or C, no balancing over a method, B0 over a bigger backbone, the current learning rate over the new one.

**Guard:** if the first baseline run scores a validation macro-F1 below 0.30 — below even the camera-only baseline's 0.326 — the model has not learned, and the notebook stops rather than waste hours. That is exactly the failure the lecturer's notebook did not catch.

A finished run leaves `runs/<name>/result.json`. If a cell is run again in the same session, finished runs are reused, not retrained.

In [ ]:
results = []
EXPERIMENTS = {
    "E1": ("Baseline transfer learning", "EfficientNetB0 224 px, variant A, no balancing",
           "Reference point for every other experiment"),
    "E2": ("Does contrast/edge enhancement help?", "Variant B (CLAHE + unsharp) or C (Ben Graham) instead of A",
           "The brief asks for enhancement; published evidence is mixed"),
    "E3": ("Does class balancing help the rare stages?", "Class weights or oversampling instead of none",
           "Severe and Proliferative are 5-8% of the images"),
    "E4": ("Is fine-tuning worth it, and at what learning rate?",
           "Phase 1 (frozen backbone) vs phase 2 (top 25% fine-tuned), every run; "
           "fine-tuning learning rate 1e-5 vs 1e-4 on the final configuration",
           "Frozen vs fine-tuned costs no extra runs; the learning rate decides how far the pretrained weights move"),
    "E5": ("Is a different backbone better?", "EfficientNetB3 at 300 px; MobileNetV2 at 224 px",
           "Higher resolution for small lesions; a lighter, familiar network"),
}

def run_config(experiment, tag, backbone="EfficientNetB0", variant="A", balance="none", ft_lr=None):
    """Train one configuration once per seed (each run in its own process) and collect the results.
    `ft_lr` overrides the fine-tuning learning rate (default: T.FT_LR)."""
    cache = cache_variant(variant, T.SIZES[backbone])
    for seed in SEEDS[backbone]:
        name = f"{tag}_s{seed}"
        done = RUN_DIR / name / "result.json"
        if done.exists():
            print(f"--- {name}: already finished, reusing its result")
        else:
            run_child({"experiment": experiment, "config": tag, "name": name, "backbone": backbone,
                       "variant": variant, "balance": balance, "seed": seed, "run_dir": str(RUN_DIR / name),
                       "cache": str(cache), "splits_csv": str(SPLITS_CSV), "mixed_camera": MIXED_CAMERA,
                       **({"ft_lr": ft_lr} if ft_lr else {})})
        results.append(json.loads(done.read_text()))
        pd.DataFrame(results).to_csv(OUT_DIR / "experiment_log.csv", index=False)   # saved after every run

def summary():
    """One row per configuration: the mean over its seeds, and the lowest and highest macro-F1."""
    table = pd.DataFrame(results)
    grouped = table.groupby("config", sort=False)
    out = grouped.agg(experiment=("experiment", "first"), backbone=("backbone", "first"),
                      variant=("variant", "first"), balance=("balance", "first"), ft_lr=("ft_lr", "first"),
                      runs=("seed", "size"),
                      macro_f1=("macro_f1", "mean"), macro_f1_min=("macro_f1", "min"),
                      macro_f1_max=("macro_f1", "max"), qwk=("qwk", "mean"), accuracy=("accuracy", "mean"),
                      dr_accuracy=("dr_accuracy", "mean"), dr_sensitivity=("dr_sensitivity", "mean"),
                      dr_specificity=("dr_specificity", "mean"),
                      phase1_macro_f1=("phase1_best_macro_f1", "mean"),
                      macro_f1_argmax=("macro_f1_argmax", "mean"), minutes=("minutes", "sum"))
    if "mixed_camera_macro_f1" in table:
        out["mixed_camera_macro_f1"] = grouped["mixed_camera_macro_f1"].mean()
        out["mixed_camera_dr_accuracy"] = grouped["mixed_camera_dr_accuracy"].mean()
    return out

def pick(candidates, simplest):
    """The winner among `candidates`: highest mean macro-F1, unless its lead over the simplest
    option is within the seed-to-seed spread - then the simplest option is kept."""
    table = summary().loc[candidates]
    best = table["macro_f1"].idxmax()
    spread = float((table["macro_f1_max"] - table["macro_f1_min"]).max())
    lead = float(table.loc[best, "macro_f1"] - table.loc[simplest, "macro_f1"])
    if best != simplest and lead <= spread:
        print(f"   {best} leads by {lead:.3f}, within the seed spread {spread:.3f} -> keep {simplest}")
        return simplest
    print(f"   winner: {best} (lead {lead:.3f}, seed spread {spread:.3f})")
    return best

def show(candidates):
    columns = ["macro_f1", "macro_f1_min", "macro_f1_max", "qwk", "accuracy", "dr_accuracy", "phase1_macro_f1"]
    print(summary().loc[candidates, columns].round(3).to_string())

In [ ]:
# E1 + E2 - the baseline and the three preprocessing variants
run_config("E1", "E1_A")
first = results[0]["macro_f1"]
assert first >= MIN_BASELINE_F1, (
    f"the baseline's validation macro-F1 is {first:.3f}, below {MIN_BASELINE_F1} - the model is not "
    "learning. Look at runs/E1_A_s42/curves.png before running anything else.")
run_config("E2", "E2_B", variant="B")
run_config("E2", "E2_C", variant="C")
show(["E1_A", "E2_B", "E2_C"])
winner = pick(["E1_A", "E2_B", "E2_C"], simplest="E1_A")
BEST_VARIANT = summary().loc[winner, "variant"]
print("variant carried forward:", BEST_VARIANT)

In [ ]:
# E3 - class balancing, on the winning variant
no_balance = {"A": "E1_A", "B": "E2_B", "C": "E2_C"}[BEST_VARIANT]
run_config("E3", f"E3_{BEST_VARIANT}_weights", variant=BEST_VARIANT, balance="weights")
run_config("E3", f"E3_{BEST_VARIANT}_oversample", variant=BEST_VARIANT, balance="oversample")
candidates = [no_balance, f"E3_{BEST_VARIANT}_weights", f"E3_{BEST_VARIANT}_oversample"]
show(candidates)
BEST_B0 = pick(candidates, simplest=no_balance)
BEST_BALANCE = summary().loc[BEST_B0, "balance"]
print("balancing carried forward:", BEST_BALANCE)

In [ ]:
# E5 - other backbones, with the winning variant and balancing
run_config("E5", "E5_MobileNetV2", backbone="MobileNetV2", variant=BEST_VARIANT, balance=BEST_BALANCE)
run_config("E5", "E5_EfficientNetB3", backbone="EfficientNetB3", variant=BEST_VARIANT, balance=BEST_BALANCE)
candidates = [BEST_B0, "E5_MobileNetV2", "E5_EfficientNetB3"]
show(candidates)
FINAL_CONFIG = pick(candidates, simplest=BEST_B0)
print("configuration carried forward:", FINAL_CONFIG)

In [ ]:
# E4 (second part) - the fine-tuning learning rate, tried on the final configuration
final = summary().loc[FINAL_CONFIG]
run_config("E4", "E4_ft_lr_1e-5", backbone=final["backbone"], variant=final["variant"],
           balance=final["balance"], ft_lr=1e-5)
candidates = [FINAL_CONFIG, "E4_ft_lr_1e-5"]
show(candidates)
FINAL_CONFIG = pick(candidates, simplest=FINAL_CONFIG)
print("final configuration:", FINAL_CONFIG)

## 8. Results

Every number below is on the **validation** split. The dashed red line in figure 22 is the camera-only model from notebook 2, which never looks at the retina; the dotted grey line is "always No DR". A CNN result means little until it is read against the camera-only line.

In [ ]:
table = summary()
table.to_csv(OUT_DIR / "experiment_summary.csv")
pd.DataFrame([{"experiment": k, "objective": v[0], "what_changed": v[1], "why": v[2]}
              for k, v in EXPERIMENTS.items()]).to_csv(OUT_DIR / "experiment_record.csv", index=False)
shown = table.round(3)
shown["ft_lr"] = table["ft_lr"].map("{:g}".format)             # 0.0001 would round to 0.0
print(shown.to_string())

baselines = pd.read_csv(find_file("baselines_val.csv"), index_col=0)
camera_only = baselines.loc["camera only (image size)"]
always_no_dr = baselines.loc["always 'No DR'"]

# Figure 22 - every configuration against the two models that never look at the retina
order = list(table.index)
fig, axes = plt.subplots(1, 3, figsize=(16, 0.55 * len(order) + 2.2), sharey=True)
for ax, metric, label in ((axes[0], "macro_f1", "Macro-F1"), (axes[1], "qwk", "QWK"),
                          (axes[2], "dr_accuracy", "DR yes/no accuracy")):
    y = np.arange(len(order))
    values = table.loc[order, metric].to_numpy()
    ax.barh(y, values, height=0.6, color=BLUE)
    if metric == "macro_f1":                                   # the seed spread as whiskers
        low, high = table.loc[order, "macro_f1_min"].to_numpy(), table.loc[order, "macro_f1_max"].to_numpy()
        ax.errorbar(values, y, xerr=[values - low, high - values], fmt="none", ecolor=INK, capsize=3,
                    linewidth=1)
    for yi, v in zip(y, values):
        ax.text(v + 0.012, yi, f"{v:.3f}", va="center", fontsize=8.5, color=INK_SOFT)
    ax.axvline(camera_only[metric], color=FLAG, linestyle="--", linewidth=1.3)
    ax.axvline(always_no_dr[metric], color=MUTED, linestyle=":", linewidth=1.3)
    ax.set_xlim(0, 1.12)
    ax.set_title(f"{label}  (camera only: {camera_only[metric]:.2f})")
    tidy(ax, grid_axis="x")
axes[0].set_yticks(np.arange(len(order)))
axes[0].set_yticklabels([f"{c}  ({int(table.loc[c, 'runs'])} run{'s' if table.loc[c, 'runs'] > 1 else ''})"
                         for c in order])
axes[0].invert_yaxis()
fig.suptitle("Validation scores of every configuration (dashed red: camera only; dotted grey: always 'No DR')",
             fontsize=12, fontweight="bold", color=INK)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.savefig(FIG_DIR / "fig22_experiments.png")
plt.show()

In [ ]:
# Figure 23 - E4: frozen backbone (phase 1) against fine-tuned (final)
fig, ax = plt.subplots(figsize=(9, 0.5 * len(order) + 1.8))
y = np.arange(len(order))
frozen, tuned = table.loc[order, "phase1_macro_f1"].to_numpy(), table.loc[order, "macro_f1"].to_numpy()
ax.hlines(y, frozen, tuned, color=GRID, linewidth=3)
ax.scatter(frozen, y, s=60, color=MUTED, zorder=3, label="phase 1: backbone frozen")
ax.scatter(tuned, y, s=60, color=BLUE, zorder=3, label="final: top 25% fine-tuned")
for yi, a, b in zip(y, frozen, tuned):                       # the change, just right of the pair
    ax.annotate(f"{b - a:+.3f}", (max(a, b), yi), xytext=(9, 0), textcoords="offset points",
                va="center", fontsize=8.5, color=INK_SOFT)
low, high = min(frozen.min(), tuned.min()), max(frozen.max(), tuned.max())
ax.set_xlim(low - 0.05, high + 0.08)
ax.set_yticks(y)
ax.set_yticklabels(order)
ax.invert_yaxis()
ax.set_xlabel("Validation macro-F1")
ax.set_title(f"Fine-tuning changes macro-F1 by {np.mean(tuned - frozen):+.3f} on average")
ax.legend(frameon=False, labelcolor=INK_SOFT, ncols=2, loc="upper center", bbox_to_anchor=(0.5, -0.14))
tidy(ax, grid_axis="x")
plt.savefig(FIG_DIR / "fig23_frozen_vs_finetuned.png")
plt.show()

In [ ]:
# The final model: curves, confusion matrix, and scores per camera (all on validation)
final_runs = pd.DataFrame(results).query("config == @FINAL_CONFIG")
final_run = final_runs.loc[final_runs["macro_f1"].idxmax()]
final_dir = RUN_DIR / final_run["name"]
print("final model:", final_run["name"])

# Figure 24 - training and validation curves of the final model, across both phases
history = pd.read_csv(final_dir / "history.csv")
T.plot_curves(history, int((history["phase"] == 1).sum()),
              f"Final model {final_run['name']}: training vs validation",
              FIG_DIR / "fig24_final_curves.png", show=True)

# Figure 25 - confusion matrix on validation (rows: true stage; columns: the model's answer)
preds = pd.read_csv(final_dir / "val_predictions.csv")
matrix = confusion_matrix(preds["true"], preds["pred"], labels=range(5))
row_share = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(6.6, 5.6))
ax.imshow(row_share, cmap="Blues", vmin=0, vmax=1)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{matrix[i, j]}\n{100 * row_share[i, j]:.0f}%", ha="center", va="center",
                fontsize=9, color="white" if row_share[i, j] > 0.55 else INK)
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels([f"{i} {CLASS_NAMES[i]}" for i in range(5)], rotation=30, ha="right")
ax.set_yticklabels([f"{i} {CLASS_NAMES[i]}" for i in range(5)])
ax.set_xlabel("Predicted stage")
ax.set_ylabel("True stage")
ax.set_title(f"Validation confusion matrix - {final_run['name']}")
plt.savefig(FIG_DIR / "fig25_final_confusion_val.png")
plt.show()

# Scores per camera: does the model do more than recognise the camera?
# (QWK is undefined - NaN - for a camera whose validation images are all one stage)
import warnings
preds = preds.merge(splits[["id_code", "camera_group"]], on="id_code")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    by_camera = pd.DataFrame({camera: {"images": len(g), **T.scores(g["true"], g["pred"])}
                              for camera, g in preds.groupby("camera_group")}).T
by_camera.to_csv(OUT_DIR / "final_val_by_camera.csv")
print("\nfinal model on validation, per camera:")
print(by_camera.round(3).to_string())
print(f"\ntwo-step answer vs plain highest probability (macro-F1): "
      f"{final_run['macro_f1']:.3f} vs {final_run['macro_f1_argmax']:.3f}")

## 9. Save the results

- **`best_model.keras`** — the final model (a copy of its best checkpoint), for notebook 4 and the app.
- **`best_config.json`** — which configuration it is, with its validation scores.
- **`experiment_log.csv`** (one row per run), **`experiment_summary.csv`** (one row per configuration) and **`experiment_record.csv`** (the objective, change and reason of each experiment).
- **`runs/`** — every run's checkpoint, history, validation predictions and curves.
- **`dr_training.py`** — the training code.
- **`report_bundle.zip`** — the figures and tables only, a few MB, for your report. **Download this file, not the whole output.**

In [ ]:
shutil.copy(final_dir / "best.keras", OUT_DIR / "best_model.keras")
best_config = {k: (v.item() if hasattr(v, "item") else v) for k, v in final_run.to_dict().items()}
best_config.update({"image_size": int(T.SIZES[final_run["backbone"]]), "weights": T.WEIGHTS,
                    "unfreeze_fraction": T.UNFREEZE_FRACTION, "dropout": T.DROPOUT,
                    "head_lr": T.HEAD_LR, "fine_tune_lr": float(final_run["ft_lr"]),
                    "decision_rule": "DR if P(stages 1-4) >= 0.5, then the most likely of stages 1-4"})
with open(OUT_DIR / "best_config.json", "w") as f:
    json.dump(best_config, f, indent=2, default=str)

with zipfile.ZipFile(OUT_DIR / "report_bundle.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(FIG_DIR.glob("*.png")):
        bundle.write(path, f"figures/{path.name}")
    for name in ("experiment_log.csv", "experiment_summary.csv", "experiment_record.csv",
                 "final_val_by_camera.csv", "best_config.json", "dr_training.py"):
        bundle.write(OUT_DIR / name, name)
    for run in sorted(RUN_DIR.iterdir()):
        for name in ("curves.png", "history.csv", "result.json"):
            if (run / name).exists():
                bundle.write(run / name, f"runs/{run.name}/{name}")
print(json.dumps(best_config, indent=2, default=str))
print(f"\nreport_bundle.zip: {(OUT_DIR / 'report_bundle.zip').stat().st_size / 1e6:.1f} MB")

## 10. What to do next

1. When the saved version has finished, download **`report_bundle.zip`** only — the full output includes every model — and unzip it into `CW1_DR_Stage_Detection/results/notebook3/`.
2. Send back `experiment_summary.csv`, `best_config.json` and figures 22–25.
3. Notebook 4 then evaluates `best_model.keras` **once** on the untouched test split: all the metrics, per-camera results, error analysis and Grad-CAM.